# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates loading and exploring the FAIR^2 dataset package using the [mlcroissant](https://mlcroissant.readthedocs.io/) library. It guides you step by step through extracting metadata, record sets, and performing exploratory data analysis using record set, field, and column `@id` values throughout.

### Dataset Source
The dataset follows the Croissant schema and is accessible at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Install mlcroissant if needed
!pip install mlcroissant

## 1. Data Loading

Load metadata and record sets from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access top-level metadata as an object
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Version: {metadata.version}")
print(f"Identifier: {getattr(metadata, 'identifier', None)}")

## 2. Data Overview

Review the available record sets and their fields, using only `@id` for all data entities as prescribed by the Croissant schema.

In [ ]:
# List all record sets and their fields by @id
record_sets = list(dataset.record_sets)

print("Available record sets and their fields (@id):\n")

record_set_ids = []
for rs in record_sets:
    print(f"- Record Set Name: {getattr(rs, 'name', 'Unnamed')} (@id: {rs.id})")
    record_set_ids.append(rs.id)
    if hasattr(rs, 'fields') and rs.fields:
        for f in rs.fields:
            print(f"    - Field Name: {getattr(f, 'name', 'Unnamed')} (@id: {f.id})")
    else:
        print("    [No fields present]")
    print("") # Add a blank line for readability

print(f"All record set @id's: {record_set_ids}")

## 3. Data Extraction

Load the data for each record set into a Pandas DataFrame using only `@id` references from the overview above.

In [ ]:
# Use all available record set @ids obtained above for extraction
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for record set @id: {record_set_id}. Shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    print("")

# Show first rows for the FIRST record set as an example
if len(record_set_ids) > 0:
    display(dataframes[record_set_ids[0]].head())

## 4. Exploratory Data Analysis (EDA)

Let's perform some common exploratory and preprocessing tasks. We'll select a numeric field from one record set for this demonstration and process it using only its `@id` for all references.

In [ ]:
# For demonstration, select the first available record set and numeric field
chosen_record_set_id = record_set_ids[0]
chosen_df = dataframes[chosen_record_set_id]

print(f"Number of records in {chosen_record_set_id}: {len(chosen_df)}")

# Try to find numeric fields by dtype or by examining the DataFrame
numeric_field_id = None
for col in chosen_df.columns:
    if pd.api.types.is_numeric_dtype(chosen_df[col]):
        numeric_field_id = col
        break

if numeric_field_id is None:
    print(f"No numeric fields found in record set {chosen_record_set_id}.")
else:
    print(f"Using numeric field: {numeric_field_id}")

    # Basic filtering example: filter by mean
    threshold = chosen_df[numeric_field_id].mean()
    filtered_df = chosen_df[chosen_df[numeric_field_id] > threshold].copy()
    print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize the numeric field (z-score) for filtered records
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} (column: {norm_col}):")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # Try to group by a categorical field (use the first object/ecologically string-type field if available)
    group_field_id = None
    for col in chosen_df.columns:
        if chosen_df[col].dtype == 'object' and col != numeric_field_id:
            group_field_id = col
            break

    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())
    else:
        print("No categorical field found for grouping.")

## 5. Visualization

Visualize the distribution of the selected numeric field and its relationship with a categorical group (if extracted).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(7, 4))
    sns.histplot(chosen_df[numeric_field_id].dropna(), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id} in {chosen_record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id:
        plt.figure(figsize=(10, 5))
        sns.boxplot(data=filtered_df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id} (filtered records)")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=30)
        plt.show()

## 6. Conclusion

In this notebook, we demonstrated loading a FAIR-compliant clinical dataset using the `mlcroissant` library, inspecting available record sets and fields by their `@id`, loading and manipulating the data, and visualizing basic distributions. All operations referenced only Croissant entity `@id`s for maximum clarity and reproducibility. 

- The dataset contains comprehensive clinicopathological variables on second primary colorectal cancer in survivors.
- We showed how to load all record sets, extract and normalize numeric fields, and group results by categorical fields using only `@id`s as references.
- This workflow can be extended for more advanced analysis, model training, or further FAIR workflows using the Croissant schema.

**Next steps:** Consider examining additional record sets, performing more detailed statistical analysis, or integrating this pipeline into larger research or clinical data flows.